In [9]:
import polars as pl 
import polars.selectors as cs

In [6]:

df_id_train = pl.scan_csv("../Data/Raw/train_identity.csv")
df_tr_train = pl.scan_csv("../Data/Raw/train_transaction.csv")

join_train = df_tr_train.join(df_id_train,on="TransactionID",how="left")
join_train.sink_parquet("../Data/Raw/train.parquet")

df_id_test = pl.scan_csv("../Data/Raw/test_identity.csv")
df_tr_test = pl.scan_csv("../Data/Raw/test_transaction.csv")

join_test = df_tr_test.join(df_id_test,on="TransactionID",how="left")
join_test.sink_parquet("../Data/Raw/Test.parquet")

In [10]:
df_train = pl.scan_parquet("../Data/Raw/train.parquet")
df_test = pl.scan_parquet("../Data/Raw/Test.parquet")
print(df_train.head(5).collect().glimpse())

Rows: 5
Columns: 434
$ TransactionID  <i64> 2987176, 2987178, 2987180, 2987190, 2987194
$ isFraud        <i64> 0, 0, 0, 0, 0
$ TransactionDT  <i64> 89283, 89372, 89404, 89565, 89624
$ TransactionAmt <f64> 39.0, 445.0, 83.95, 32.325, 1486.44
$ ProductCD      <str> 'W', 'W', 'W', 'C', 'W'
$ card1          <i64> 15066, 14290, 4348, 5458, 11556
$ card2          <f64> 170.0, 512.0, 147.0, 266.0, 309.0
$ card3          <f64> 150.0, 150.0, 150.0, 185.0, 150.0
$ card4          <str> 'mastercard', 'visa', 'visa', 'mastercard', 'visa'
$ card5          <f64> 102.0, 226.0, 195.0, 224.0, 226.0
$ card6          <str> 'credit', 'debit', 'debit', 'credit', 'debit'
$ addr1          <f64> 204.0, 225.0, 205.0, null, 181.0
$ addr2          <f64> 87.0, 87.0, 87.0, null, 87.0
$ dist1          <f64> null, 11.0, null, null, null
$ dist2          <f64> null, null, null, null, null
$ P_emaildomain  <str> null, 'gmail.com', 'yahoo.com', 'hotmail.com', 'comcast.net'
$ R_emaildomain  <str> null, null, null, 'hotma

In [11]:
print(df_train.describe())

shape: (9, 435)
┌────────────┬─────────────┬──────────┬────────────┬───┬────────┬────────┬────────────┬────────────┐
│ statistic  ┆ Transaction ┆ isFraud  ┆ Transactio ┆ … ┆ id_37  ┆ id_38  ┆ DeviceType ┆ DeviceInfo │
│ ---        ┆ ID          ┆ ---      ┆ nDT        ┆   ┆ ---    ┆ ---    ┆ ---        ┆ ---        │
│ str        ┆ ---         ┆ f64      ┆ ---        ┆   ┆ str    ┆ str    ┆ str        ┆ str        │
│            ┆ f64         ┆          ┆ f64        ┆   ┆        ┆        ┆            ┆            │
╞════════════╪═════════════╪══════════╪════════════╪═══╪════════╪════════╪════════════╪════════════╡
│ count      ┆ 590540.0    ┆ 590540.0 ┆ 590540.0   ┆ … ┆ 140985 ┆ 140985 ┆ 140810     ┆ 118666     │
│ null_count ┆ 0.0         ┆ 0.0      ┆ 0.0        ┆ … ┆ 449555 ┆ 449555 ┆ 449730     ┆ 471874     │
│ mean       ┆ 3282269.5   ┆ 0.03499  ┆ 7.3723e6   ┆ … ┆ null   ┆ null   ┆ null       ┆ null       │
│ std        ┆ 170474.3583 ┆ 0.183755 ┆ 4.6172e6   ┆ … ┆ null   ┆ null   ┆ 

In [14]:
train_pipeline = (
    df_train.with_columns(
        cs.string().fill_null("unknown")
    )
    .with_columns(
        cs.string().cast(pl.Categorical)
    )
    .drop(["TransactionID"])
)

test_pipeline = (
    df_test.with_columns(
        cs.string().fill_null("unknown")
    )
    .with_columns(
        cs.string().cast(pl.Categorical)
    )
    .drop(["TransactionID"])
)

In [17]:
train_pipeline.sink_parquet("../Data/Processed/train_ready.parquet")
test_pipeline.sink_parquet("../Data/Processed/test_ready.parquet")